🔹 Step 1: Load the dataset


In [16]:
import json
import pandas as pd

# Load the synthetic SMS dataset
with open("data/sms_messages.json", "r") as f:
    sms_data = json.load(f)

# View first 3 SMS messages
for i in range(3):
    print(sms_data[i]["text"])


331.33 debited from AC XXXXXXXX6545 as POS TXN on 30 Mar 2025 18:28 at Domino's. Avl Bal 2021.74 Call 94112448888 for info
48.23 debited from AC XXXXXXXX6545 as POS TXN on 16 Mar 2025 02:27 at Bookstore. Avl Bal 1969.34 Call 94112448888 for info
212.18 debited from AC XXXXXXXX6545 as POS TXN on 07 Mar 2025 00:54 at Domino's. Avl Bal 4947.53 Call 94112448888 for info


🔹 Step 9: Save Forecast to JSON or CSV

🔹 Step 2: Parse SMS into structured features


In [17]:
import re
from datetime import datetime

# Parse the SMS text into structured data
parsed_data = []

for sms in sms_data:
    text = sms["text"]
    
    # Extract amount
    amount_match = re.search(r"([\d,.]+)\s+debited", text)
    amount = float(amount_match.group(1).replace(",", "")) if amount_match else None

    # Extract datetime
    datetime_match = re.search(r"on (\d{2} \w{3} \d{4} \d{2}:\d{2})", text)
    dt = datetime.strptime(datetime_match.group(1), "%d %b %Y %H:%M") if datetime_match else None

    # Extract merchant
    merchant_match = re.search(r"at ([^.]+)\.", text)
    merchant = merchant_match.group(1).strip() if merchant_match else "Unknown"

    # Append structured row
    parsed_data.append({
        "amount": amount,
        "datetime": dt,
        "hour": dt.hour if dt else None,
        "day_of_week": dt.strftime("%A") if dt else None,
        "merchant": merchant
    })

df = pd.DataFrame(parsed_data)
df.head()


,amount,datetime,hour,day_of_week,merchant
0,331.33,2025-03-30 18:28:00,18,Sunday,Domino's
1,48.23,2025-03-16 02:27:00,2,Sunday,Bookstore
2,212.18,2025-03-07 00:54:00,0,Friday,Domino's
3,346.18,2025-03-27 16:05:00,16,Thursday,Keells Super
4,323.39,2025-03-01 22:48:00,22,Saturday,Spar Supermarket


🔹 Step 3: Encode categorical values + prepare features


In [18]:
from sklearn.preprocessing import LabelEncoder

# Encode 'merchant' into numeric
merchant_encoder = LabelEncoder()
df["merchant_encoded"] = merchant_encoder.fit_transform(df["merchant"])

# Map day of week to number (Monday=0, ..., Sunday=6)
df["day_num"] = df["day_of_week"].map({
    "Monday": 0, "Tuesday": 1, "Wednesday": 2,
    "Thursday": 3, "Friday": 4, "Saturday": 5, "Sunday": 6
})

# Final feature set
features = df[["amount", "hour", "day_num", "merchant_encoded"]]
features.head()


,amount,hour,day_num,merchant_encoded
0,331.33,18,6,5
1,48.23,2,6,3
2,212.18,0,4,5
3,346.18,16,3,11
4,323.39,22,5,18


🔹 Step 4: Train Isolation Forest


In [19]:
from sklearn.ensemble import IsolationForest

# Train model
model = IsolationForest(contamination=0.07, random_state=42)
df["is_anomaly"] = model.fit_predict(features)

# Convert -1 (outlier) to True, 1 (inlier) to False
df["is_anomaly"] = df["is_anomaly"] == -1
df["is_anomaly"].value_counts()


is_anomaly
False    46
True      4
Name: count, dtype: int64

🔹 Step 5: Save flagged results to JSON

In [20]:
# Add anomaly flag back to SMS messages
for i, row in df.iterrows():
    sms_data[i]["is_anomaly"] = bool(row["is_anomaly"])

# Save to new JSON file
with open("data/anomaly_detection/flagged_sms_transactions.json", "w") as f:
    json.dump(sms_data, f, indent=2)

print("✅ Anomaly detection results saved to flagged_sms_transactions.json")


✅ Anomaly detection results saved to flagged_sms_transactions.json
